In [75]:
import pandas as pd

In [76]:
def bondpricer(coupon: float, yield_to_maturity: float, periods: int, future_value: float) -> float:

    price = 0.0 

    for i in range(1, periods + 1):
        print(i)
        price += coupon / ((1 + yield_to_maturity) ** i)

    price += future_value / ((1 + yield_to_maturity) ** periods)
    return price

In [77]:
print(bondpricer(50, 0.06, 5, 1000))

1
2
3
4
5
957.8763621443427


In [78]:
df = pd.read_csv("fixed_income.csv")

In [79]:
df.head(1)

,Unnamed: 0,Financial Instrument,Coupon,Maturity,Ratings,Issuer Debt/Equity,Issue Date,Time-To-Maturity (TTM),Payment Frequency,Ask,Ask Yield,Duration %,Convexity,Bid,Unnamed: 13,Instrument Type
0,0,SMMSBK CORP CD 82869ADY8,5.45,Jul25'24,0,0,Oct 25 '23,0.01,Monthly,100.005,3.511,0.005479452200233936,0.0000003,99.851,0.0,Certificates of Deposit


In [80]:
df.dtypes

Unnamed: 0                  int64
Financial Instrument       object
Coupon                    float64
Maturity                   object
Ratings                    object
Issuer Debt/Equity         object
Issue Date                 object
Time-To-Maturity (TTM)    float64
Payment Frequency          object
Ask                       float64
Ask Yield                 float64
Duration %                 object
Convexity                  object
Bid                        object
Unnamed: 13               float64
Instrument Type            object
dtype: object

In [81]:
# Standardise column names

df.columns = (
    df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_", regex = False)
        .str.replace(":", "", regex = False)
        .str.replace("/", "_", regex = False)
        .str.replace("%", "_pct")
        .str.replace("-", "_", regex = False)
)

In [82]:
df.isna().mean().sort_values(ascending = False)

unnamed_0                 0.0
financial_instrument      0.0
coupon                    0.0
maturity                  0.0
ratings                   0.0
issuer_debt_equity        0.0
issue_date                0.0
time_to_maturity_(ttm)    0.0
payment_frequency         0.0
ask                       0.0
ask_yield                 0.0
duration__pct             0.0
convexity                 0.0
bid                       0.0
unnamed_13                0.0
instrument_type           0.0
dtype: float64

In [83]:
object_cols = df.select_dtypes(include = "object").columns

In [84]:
num_cols = df.select_dtypes(include = "number").columns
num_cols

Index(['unnamed_0', 'coupon', 'time_to_maturity_(ttm)', 'ask', 'ask_yield',
       'unnamed_13'],
      dtype='object')

In [85]:
type(df["time_to_maturity_(ttm)"].dtype)

numpy.dtypes.Float64DType

In [86]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} has taken {end - start} seconds")
        return result
    return wrapper
    

In [100]:
datetime_cols = ["maturity", "issue_date"]

def convert_to_datetime(df: pd.DataFrame, cols = list[str]) -> pd.DataFrame:

    for col in cols:
        
        df[col] = pd.to_datetime(df[col]).astype("datetime64[ns]")
    
    return df

def convert_to_datetime_pyarrow(df: pd.DataFrame, cols = list[str]) -> pd.DataFrame:

    for col in cols:
        
        df[col] = pd.to_datetime(df[col]).astype("timestamp[ns][pyarrow]")
    
    return df

In [101]:
df_copy = df.copy()

df_datetime = convert_to_datetime(df_copy.copy(), datetime_cols)

df_datetime_pyarrow = convert_to_datetime_pyarrow(df_copy.copy(), datetime_cols)

C:\Users\aleks\AppData\Local\Temp\ipykernel_24280\3248282586.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col]).astype("datetime64[ns]")
C:\Users\aleks\AppData\Local\Temp\ipykernel_24280\3248282586.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col]).astype("datetime64[ns]")
C:\Users\aleks\AppData\Local\Temp\ipykernel_24280\3248282586.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col]).astype("timestamp[ns][pyarrow]")
C:\Users\aleks\AppData\Local\Temp\ipykernel_24280\32482

In [102]:
df_datetime.dtypes

unnamed_0                          int64
financial_instrument              object
coupon                           float64
maturity                  datetime64[ns]
ratings                           object
issuer_debt_equity                object
issue_date                datetime64[ns]
time_to_maturity_(ttm)           float64
payment_frequency                 object
ask                              float64
ask_yield                        float64
duration__pct                     object
convexity                         object
bid                               object
unnamed_13                       float64
instrument_type                   object
dtype: object

In [103]:
df_datetime_pyarrow.dtypes

unnamed_0                                  int64
financial_instrument                      object
coupon                                   float64
maturity                  timestamp[ns][pyarrow]
ratings                                   object
issuer_debt_equity                        object
issue_date                timestamp[ns][pyarrow]
time_to_maturity_(ttm)                   float64
payment_frequency                         object
ask                                      float64
ask_yield                                float64
duration__pct                             object
convexity                                 object
bid                                       object
unnamed_13                               float64
instrument_type                           object
dtype: object

In [107]:
@timer
def test(df: pd.DataFrame, col: str) -> str:
    
    
    year = df[col].dt.year

    return year
    
test1 = test(df_datetime, "maturity")
test2 = test(df_datetime_pyarrow, "maturity")

test has taken 0.00038350000249920413 seconds
test has taken 0.01186519999828306 seconds


In [3]:
def dirty_price(present_value: float, YTM: float, no_of_payments: int, t: float, T: float) -> float:
    return present_value * ( 1 + (YTM / no_of_payments) )**(t/T)

In [4]:
dirty_price(1027.84, 0.05, 1, 50, 365)

1034.732663304513